In [1]:
import pandas as pd
import openml

In [ ]:
import openml
import pandas as pd

# Konfiguracja wyświetlania
pd.set_option('display.max_rows', 100)

def get_best_models_simple():
    print("1. Pobieranie zadań i wyników...")
    tasks = openml.tasks.list_tasks(tag='OpenML-CC18', output_format='dataframe')
    binary_ids = tasks[tasks['NumberOfClasses'] == 2]['tid'].tolist()
    
    # Pobieramy więcej wyników, żeby mieć punkt odniesienia (max)
    evals = openml.evaluations.list_evaluations(
        function='area_under_roc_curve', 
        tasks=binary_ids,
        size=200000, 
        output_format='dataframe'
    )

    if evals.empty:
        print("Brak wyników.")
        return

    # === NAJPROSTSZA NORMALIZACJA ===
    
    # Krok 1: Znajdź MAX wynik dla każdego zadania (task_id)
    # Tworzymy kolumnę, która w każdym wierszu ma informację, jaki był rekord na tym zbiorze
    evals['max_for_task'] = evals.groupby('task_id')['value'].transform('max')
    
    # Krok 2: Oblicz prosty stosunek: Mój Wynik / Rekord
    # 1.00 = to był najlepszy model na tym zbiorze
    # 0.95 = był bardzo blisko najlepszego
    evals['score_ratio'] = evals['value'] / evals['max_for_task']

    # Krok 3: Agregacja
    ranking = evals.groupby('flow_id').agg(
        final_score=('score_ratio', 'mean'), # Średnia z "procentów mistrza"
        count=('task_id', 'count')           # Na ilu zbiorach testowano
    ).reset_index()

    # Krok 4: Odsiewamy modele przetestowane rzadziej niż 5 razy (kluczowe!)
    
    # Sortujemy: Najlepsze na górze
    ranking = ranking.sort_values(by='final_score', ascending=False)

    print("\n=== TOP 10 MODELI (Wg 'bliskości ideału') ===")
    print(ranking.head(10))

    # Pobierz nazwę zwycięzcy
    top_id = ranking.iloc[0]['flow_id']
    try:
        name = openml.flows.get_flow(top_id).name
        print(f"\nZwycięzca (ID: {top_id}) to: {name}")
    except:
        pass

if __name__ == "__main__":
    get_best_models_simple()

1. Pobieranie zadań i wyników...


In [40]:

ranking

,flow_id,median_auc,mean_relative,count,flow_name
0,57,0.666985,-0.262427,1,weka.OneR(1)
1,58,0.951281,0.021869,1,weka.NaiveBayes(1)
2,61,0.997912,0.068500,1,weka.REPTree(1)
3,62,0.666432,-0.262980,1,weka.DecisionStump(1)
4,67,0.951346,0.021934,1,weka.BayesNet_K2(1)
...,...,...,...,...,...
207,8795,0.998554,0.068927,75,sklearn.pipeline.Pipeline(columntransformer=sk...
208,8796,0.999168,0.069617,5,sklearn.pipeline.Pipeline(columntransformer=sk...
209,8797,0.998621,0.068678,77,sklearn.pipeline.Pipeline(columntransformer=sk...
210,8815,0.996560,0.067386,19,sklearn.pipeline.Pipeline(columntransformer=sk...
